In [6]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))


from scripts.text_matching import normalise_text

In [7]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
still_unmatched_file = Path(project_root / "data/people/still_unmatched.json")

In [8]:
with open(still_unmatched_file , "r") as f:
   still_unmatched = json.load(f)
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)

In [ ]:

variants_dict = {}
for variant in variants:
    variants_dict[normalise_text(variant["variant_normalised"])] = variant["person_id"]

rprint(dict(list(variants_dict.items())[:5]))

people_dict = {}

for person in people:
    last = person["family_name"]
    first = person["given_names"]

    if last is None or first is None:
        continue

    last_norm = normalise_text(last)
    first_norm = normalise_text(first)

    key = (last_norm, first_norm)
    people_dict[key] = person["person_id"]

# rprint(dict(list(people_dict.items())[:5]))

single_dict = {}

for person in people:
    single = person["single_name"]
    if not single:
        continue
    else:
        single_norm = normalise_text(single)

    single_dict[single_norm] = person["person_id"]


In [ ]:
SEPARATORS = re.compile(r"\s*;\s*|\s+/\s+|\s+und\s+")
ETC_SUFFIX = re.compile(r"\s+u\.a\.\s*$")

matched_via_variants = {}
still_unmatched_v2 = {}

for name, entries in still_unmatched.items():
    # Strip "u.a." trailing tag, then split on multi-person separators
    cleaned = ETC_SUFFIX.sub("", name).strip()
    parts = [p.strip() for p in SEPARATORS.split(cleaned) if p.strip()]

    person_ids = []
    failed_parts = []

    for part in parts:
        # Try 1: full-string variant lookup
        person_id = variants_dict.get(part)
        # rprint(f"part beginning: {part}")

        # Try 2: fall back to your existing first/last/single logic
        if not person_id:
            if ", " in part:
                last, first = part.split(", ", 1)
                rprint(f"after comma split: {last, first}")
                person_id = people_dict.get((last.strip(), first.strip()))
            else:
                split = part.rsplit(" ", 1)
                if len(split) == 2:
                    first, last = split
                    person_id = people_dict.get((last, first))
                else:
                    person_id = single_dict.get(part)

        if person_id:
            person_ids.append(person_id)
        else:
            failed_parts.append(part)

    if person_ids and not failed_parts:
        matched_via_variants[name] = {"person_ids": person_ids, "entries": entries}
    else:
        still_unmatched_v2[name] = entries

# print(f"Newly matched: {len(matched_via_variants)}")
# print(f"Still unmatched: {len(still_unmatched_v2)}")

rprint(matched_via_variants)
# with open("multi_person_entries_split.json", "w") as f:
# #     json.dump(matched_via_variants, f, ensure_ascii=False, indent=2)
# with open("still_unmatched_v2_file.json", "w") as f:
#     json.dump(still_unmatched_v2, f, ensure_ascii=False, indent=2)